# Jupyter Notebook Tiền xử lý & Phân tích (Ý)

- **Người thực hiện:** Ý
- **Danh mục:** Tất cả sản phẩm (Dữ liệu đã gộp)
- **Công cụ:** Pandas, NumPy, Plotly Express

**Mục tiêu SMART:**
1. **Phân tích ảnh hưởng của danh mục sản phẩm và phân khúc giá đến lượt bán:** Xác định mức giá "điểm ngọt" (tối ưu nhất) theo từng ngành hàng.
2. **Phân tích ảnh hưởng của giảm giá đến lượt bán:** Đối chiếu hiệu quả bán hàng giữa nhóm sản phẩm CÓ giảm giá với KHÔNG giảm giá.

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

# Đảm bảo hiển thị dạng float
pd.options.display.float_format = '{:,.2f}'.format

# 1. Đọc dữ liệu từ file xử lý
df_fact = pd.read_csv('../data/processed/fact_product_merged.csv', encoding='utf-8-sig')
dim_category = pd.read_csv('../data/processed/dim_category.csv', encoding='utf-8-sig')

# Nối bảng để lấy Tên Danh Mục (category_name) thay vì chỉ dùng mã (category_id)
df = df_fact.merge(dim_category[['category_id', 'category_name']], on='category_id', how='left')

print(f"📊 [PRE-CHECK] Dữ liệu ban đầu: {df.shape[0]} hàng, {df.shape[1]} thuộc tính")

📊 [PRE-CHECK] Dữ liệu ban đầu: 9186 hàng, 29 thuộc tính


## Tiền Xử Lý Dữ Liệu
Quá trình làm sạch dữ liệu bao gồm các bước sau:
1. **Loại bỏ dữ liệu thiếu:** Bỏ đi những sản phẩm khuyết thông tin ở cột `sold_count` vì không thể phân tích doanh số nếu không có dữ liệu lượt bán.
2. **Xác định trạng thái giảm giá:** Tạo cột `is_discounted` (Có/Không) để xem một sản phẩm có đang được giảm giá hay không, phục vụ cho Mục tiêu 2.
3. **Loại bỏ giá trị ngoại lai (Outliers):** Dùng phân vị 99% (`q99`) để lọc bỏ 1% các sản phẩm cá biệt bán được số lượng "siêu đột biến" (quá cao so với phần đông). Việc lọc này (từ ~9186 hàng ban đầu xuống còn ~8556 hàng) giúp các biểu đồ cột/hộp bên dưới không bị lệch tỷ lệ, không bị "kéo dãn" móp méo và thể hiện chính xác mặt bằng chung của thị trường.

*Ghi chú nhỏ: Ở cuối đoạn code này, hàm `.head(2)` được gọi ra chỉ để **in thử 2 dòng đầu** cho mục đích xem trước (preview) cấu trúc bảng. Toàn bộ 8556 dòng vẫn đang được giữ trong bộ nhớ để vẽ biểu đồ.*

In [14]:
# Chỉ phân tích những sản phẩm có số lượng bán hợp lệ
df = df.dropna(subset=['sold_count'])
df['sold_count'] = df['sold_count'].astype(float)

# Phân loại CÓ hay KHÔNG CÓ giảm giá
df['is_discounted'] = df['discount_percent'] > 0

# Loại bỏ top 1% outliers của sold_count để biểu đồ không bị nhiễu do 1 vài sp siêu outlier
q99 = df['sold_count'].quantile(0.99)

# Copy dữ liệu để tránh cảnh báo SettingWithCopyWarning của Pandas
df_clean = df[df['sold_count'] <= q99].copy()

print(f"[POST-CLEAN] Dữ liệu sau loại bỏ outlier: {df_clean.shape[0]} hàng")
df_clean.head(2)

[POST-CLEAN] Dữ liệu sau loại bỏ outlier: 8556 hàng


,product_id,platform_id,category_id,shop_id,product_name,product_url,price_current,price_original,discount_percent,price_ends_with_9,...,is_freeship,estimated_delivery_days,has_freeship_xtra_label,has_coinback_label,has_voucher_label,promotion_label_count,crawled_at,crawled_by,category_name,is_discounted
0,tiki_279199220,tiki,tiki_1882,tiki_1,"Nồi áp suất cơ Elmich 5L PCE-8792, công suất l...",https://tiki.vn/noi-ap-suat-co-elmich-5l-pce-8...,"1,680,000.00","3,695,000.00",54.50,False,...,False,NaN,False,False,False,0,2026-03-30T08:50:12+07:00,duong,Dien Gia Dung,True
1,tiki_278899036,tiki,tiki_1882,tiki_1,Ấm Đun Nước Siêu Tốc Hafele HS-K1707DX - 535.4...,https://tiki.vn/am-dun-nuoc-sieu-toc-hafele-hs...,"690,000.00","1,590,000.00",56.60,False,...,False,NaN,False,False,False,0,2026-03-30T08:50:14+07:00,duong,Dien Gia Dung,True


## Mục tiêu 1: Hiệu suất bán hàng theo Danh mục và Phân khúc giá

In [15]:
if 'price_bucket' in df_clean.columns and 'category_name' in df_clean.columns:
    # Nhóm theo category_name thay vì category_id
    df_agg_1 = df_clean.groupby(['category_name', 'price_bucket'])['sold_count'].mean().reset_index()
    df_agg_1 = df_agg_1.round({'sold_count': 0})
    
    # Rút gọn Tên Danh Mục nếu nó quá dài (tuỳ chọn rút gọn tên)
    df_agg_1['category_name_short'] = df_agg_1['category_name'].apply(lambda x: x[:30] + '...' if isinstance(x, str) and len(x) > 30 else x)

    fig_1 = px.bar(
        df_agg_1, 
        x="category_name_short", 
        y="sold_count", 
        color="price_bucket", 
        barmode="group",
        title="[Ý] - Lượt Bán Trung Bình Theo Danh Mục & Phân Khúc Giá (Điểm Ngọt)",
        labels={"category_name_short": "Tên Danh Mục", "sold_count": "Lượt Bán TB", "price_bucket": "Phân khúc Giá"},
        text_auto=True
    )
    # Nghiêng nhãn trục X để dễ đọc
    fig_1.update_layout(xaxis_tickangle=-45)
    fig_1.show()

### Phân tích chi tiết Mục tiêu 1
Để xác định "mức giá ngọt" một cách đáng tin cậy, ta không chỉ nhìn qua Bar Chart mà cần xem xét Bảng Pivot thống kê kết hợp 2 yếu tố cốt lõi:
- **`Số_Lượng_SP` (Nguồn cung):** Cho biết có bao nhiêu mã sản phẩm đang cùng cạnh tranh trong phân khúc đó. Nếu một phân khúc bán rất chạy nhưng chỉ có 1-2 người đăng bán (số lượng SP lẻ tẻ) thì đó có thể là do may mắn hoặc cá biệt, không mang tính xu hướng chung của thị trường.
- **`Lượt_Bán_TB` (Sức hút/Nhu cầu):** Lượt trung bình bán ra của mỗi sản phẩm trong nhóm, phản ánh đúng "điểm ngọt" về giá mà khách hàng dễ xuống tiền nhất.

Những kết quả chính:
1. **TOP 3 TỪNG NGÀNH HÀNG:** Khai thác sát vào tiêu chí "ít nhất 3 phân khúc cho mỗi danh mục" trong mục tiêu SMART. Bảng này giúp ta bóc tách cụ thể ngành hàng Mẹ & Bé cần bán giá nào, trong khi ngành Điện Thoại thì bán mức giá nào là tốt nhất!
2. Kế tiếp đó, **Ma trận nhiệt (Heatmap)** sẽ trực quan hóa toàn bộ bức tranh này bằng dải màu đậm nhạt. Càng đậm nghĩa là giá đó ở ngành hàng đó càng dễ được mua.

In [22]:
# Bảng tóm tắt: Đếm số lượng sản phẩm và Lượt bán TB theo mỗi Danh Mục (Tên) thay vì Mã x mức giá
stats_1 = df_clean.groupby(['category_name', 'price_bucket']).agg(
    Số_Lượng_SP=('product_id', 'count'),
    Lượt_Bán_TB=('sold_count', 'mean')
).reset_index()

stats_1['Lượt_Bán_TB'] = stats_1['Lượt_Bán_TB'].round(1)

# Sắp xếp phân khúc "Điểm ngọt" (Lượt bán cao nhất)
sweet_spots = stats_1.sort_values(by='Lượt_Bán_TB', ascending=False).reset_index(drop=True)

# ĐÁP ỨNG TIÊU CHÍ SMART (M): Lấy ra Top 3 phân khúc giá tốt nhất cho *MỖI* ngành hàng 
top_3_per_cat = sweet_spots.groupby('category_name').head(3).reset_index(drop=True)
top_3_per_cat.index = top_3_per_cat.index + 1
print("🎯 TOP 3 PHÂN KHÚC TỐT NHẤT THEO *TỪNG* NGÀNH HÀNG (Đáp ứng tiêu chí Đo lường của Mục tiêu 1):")
display(top_3_per_cat)

# Ma trận nhiệt biểu diễn Lượt bán trung bình
pivot_df = sweet_spots.pivot(index="category_name", columns="price_bucket", values="Lượt_Bán_TB")
fig_heat = px.imshow(pivot_df, text_auto=True, aspect="auto", 
                     title="[Ý] - Ma trận Nhiệt Lượt Bán TB theo Danh mục & Phân khúc",
                     color_continuous_scale='Blues')
fig_heat.show()

🎯 TOP 3 PHÂN KHÚC TỐT NHẤT THEO *TỪNG* NGÀNH HÀNG (Đáp ứng tiêu chí Đo lường của Mục tiêu 1):


,category_name,price_bucket,Số_Lượng_SP,Lượt_Bán_TB
1,Lam Dep & Suc Khoe,<100k,184,887.00
2,Lam Dep & Suc Khoe,100k-500k,363,663.30
3,Lam Dep & Suc Khoe,500k-1M,49,411.90
4,Nha Cua & Doi Song,500k-1M,43,408.80
5,Nha Cua & Doi Song,100k-500k,290,393.10
6,Dien Thoai - May Tinh Bang,100k-500k,5,334.20
7,Dien Gia Dung,1M-5M,293,308.90
8,Dien Gia Dung,500k-1M,248,298.50
9,Nha Cua & Doi Song,<100k,322,263.40
10,Thoi Trang Nu,500k-1M,19,247.10


## Mục tiêu 2: Hiệu quả của Giảm giá

In [20]:
# Đổi True/False thành chữ tiếng Việt cho dễ nhìn
df_clean['Trang_Thai_Giam_Gia'] = df_clean['is_discounted'].map({True: 'Có Giảm Giá', False: 'Không Giảm Giá'})

# --- BIỂU ĐỒ 1: Biểu đồ cột so sánh Trung bình (Dễ hiểu nhất) ---
discount_mean = df_clean.groupby('Trang_Thai_Giam_Gia')['sold_count'].mean().reset_index()
fig_bar = px.bar(
    discount_mean, 
    x="Trang_Thai_Giam_Gia", 
    y="sold_count", 
    color="Trang_Thai_Giam_Gia",
    title="[Ý] - Lượt Bán TRUNG BÌNH Giữa Có và Không Giảm Giá",
    labels={"Trang_Thai_Giam_Gia": "Trạng thái", "sold_count": "Lượt Bán Trung Bình"},
    text_auto='.1f'
)
fig_bar.show()

# --- BIỂU ĐỒ 2: Biểu đồ tròn biểu diễn Tổng Lượt Bán (Tỷ trọng) ---
discount_sum = df_clean.groupby('Trang_Thai_Giam_Gia')['sold_count'].sum().reset_index()
fig_pie = px.pie(
    discount_sum, 
    names="Trang_Thai_Giam_Gia", 
    values="sold_count", 
    color="Trang_Thai_Giam_Gia",
    hole=0.4, # Tạo biểu đồ Donut (Bánh vòng) cho đẹp
    title="[Ý] - Tỷ trọng TỔNG LƯỢT BÁN: Có Giảm Giá vs Không Giảm Giá"
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label', textfont_size=15)
fig_pie.show()

# --- BIỂU ĐỒ 3: Biểu đồ cột nhóm so sánh THEO TỪNG NGÀNH HÀNG (Đáp ứng tiêu chí M của mục tiêu 2) ---
cat_discount_mean = df_clean.groupby(['category_name', 'Trang_Thai_Giam_Gia'])['sold_count'].mean().reset_index()
fig_bar_cat = px.bar(
    cat_discount_mean, 
    x="category_name", 
    y="sold_count", 
    color="Trang_Thai_Giam_Gia",
    barmode="group",
    title="[Ý] - Lượt Bán TB Có/Không Giảm Giá THEO TỪNG NGÀNH HÀNG",
    labels={"category_name": "Ngành Hàng", "sold_count": "Lượt Bán Trung Bình", "Trang_Thai_Giam_Gia": "Trạng thái"},
    text_auto='.0f'
)
# Làm nghiêng chữ trục x nếu tên ngành quá dài
fig_bar_cat.update_layout(xaxis_tickangle=-45)
fig_bar_cat.show()

### Phân tích chi tiết Mục tiêu 2
Dựa vào ba biểu đồ trên, ta có thể rút ra những đánh giá quan trọng về chiến lược Khuyến mãi (Giảm giá):
1. **Theo Lượt Bán Trung Bình Tổng Thể (Biểu đồ Cột số 1):** Độ cao của cột thể hiện sự chênh lệch cực kỳ rõ rệt. Sản phẩm *Có Giảm Giá* đạt năng suất bán ra trung bình cao gấp gần 9 lần so với nhóm *Không Giảm Giá*. 
2. **Theo Tỷ Trọng Tổng Lượt Bán (Biểu đồ Bánh Vòng Donut):** Biểu đồ thể hiện mức độ chiếm lĩnh thị trường. "Phần bánh" doanh thu sinh ra trên toàn sàn rơi gần như đa số (>70%) vào nhóm sản phẩm có giảm giá, dù chúng chỉ là thiểu số về lượng bày bán.
3. **Mở rộng theo Từng Ngành Hàng (Biểu đồ Cột số 3):** Biểu đồ này đáp ứng chính xác tiêu chí đặt ra là so sánh hiệu quả giảm giá trên chi tiết từng lĩnh vực. Ta có thể thấy ở bất kỳ ngành hàng nào (từ Làm Đẹp, Mẹ & Bé, Điện Thoại...), cột màu xanh (Có Giảm Giá) luôn cao áp đảo cột màu đỏ (Không Giảm Giá). 

Để chứng minh các nhận định trực quan trên một cách chính xác (về mặt toán học), bảng số liệu thống kê (`describe()`) bên dưới sẽ cung cấp cái nhìn chi tiết hơn thông qua độ lệch chuẩn, trung vị (50%), và đo lường **tỷ lệ % tăng trưởng cụ thể**.

In [21]:
# Thống kê mô tả bằng bảng (Nhóm theo trạng thái tiếng Việt)
discount_stats = df_clean.groupby('Trang_Thai_Giam_Gia')['sold_count'].describe()

# Reset index để biến 'Trang_Thai_Giam_Gia' (đang là tên hàng) thành một cột bình thường ngang hàng với các cột khác
display_stats = discount_stats.reset_index()

# Đổi luôn tên cột sang Tiếng Việt cho đồng bộ báo cáo
display_stats.rename(columns={
    'Trang_Thai_Giam_Gia': 'Trạng Thái Khuyến Mãi',
    'count': 'Số lượng SP',
    'mean': 'Trung Bình',
    'std': 'Độ lệch chuẩn',
    'min': 'Nhỏ nhất',
    '25%': 'Phân vị 25%',
    '50%': 'Trung vị (50%)',
    '75%': 'Phân vị 75%',
    'max': 'Lớn nhất'
}, inplace=True)

display(display_stats.round(2))

# Tính chênh lệch lượt bán TB
mean_no_discount = discount_stats.loc['Không Giảm Giá', 'mean']
mean_with_discount = discount_stats.loc['Có Giảm Giá', 'mean']
if mean_no_discount > 0:
    pct_diff = ((mean_with_discount - mean_no_discount) / mean_no_discount) * 100
    print(f"🔥 KẾT LUẬN: Nếu CÓ giảm giá, lượt bán trung bình tăng {pct_diff:.2f}% so với KHÔNG giảm giá.")
else:
    print("Không có thông tin cho nhóm không giảm giá.")

,Trạng Thái Khuyến Mãi,Số lượng SP,Trung Bình,Độ lệch chuẩn,Nhỏ nhất,Phân vị 25%,Trung vị (50%),Phân vị 75%,Lớn nhất
0,Có Giảm Giá,"1,847.00",483.64,"1,337.86",0.00,6.00,36.00,211.50,"11,720.00"
1,Không Giảm Giá,"6,709.00",54.81,295.35,0.00,0.00,3.00,18.00,"9,892.00"


🔥 KẾT LUẬN: Nếu CÓ giảm giá, lượt bán trung bình tăng 782.42% so với KHÔNG giảm giá.
